# 04 · Business meaning and cross-dataset checks

Positive quantity is a generic value check. A shipped order requiring a shipment date expresses a business process. Rule IDs let operations and producers discuss stable contracts.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Inspect the source
Each JSON line is one order envelope. Preserve the original text and source filename before parsing so even corrupt lines remain accountable.


In [ ]:
ORDER_FIELDS = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "status",
    "order_date",
    "shipped_date",
    "cancellation_reason",
    "order_total",
    "seller_id",
    "country",
    "arrival_date",
]
order_schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in ORDER_FIELDS]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)


def read_orders(path):
    # Preserve one source envelope per physical JSON line, including malformed JSON.
    # Source row IDs are materialized before branching; raw_text supports replay.
    raw = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_text")
        .withColumn("source_file", F.input_file_name())
        .withColumn("source_row_id", F.monotonically_increasing_id())
    )
    parsed = raw.withColumn(
        "parsed",
        F.from_json(
            "raw_text",
            order_schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    return parsed.select("source_row_id", "source_file", "raw_text", "parsed.*").cache()


orders = read_orders(f"{RAW_PATH}/orders")
orders.count()  # materialize once before splitting
customers = spark.read.option("header", True).csv(f"{RAW_PATH}/customers")
products = spark.read.option("header", True).csv(f"{RAW_PATH}/products")
items = spark.read.option("header", True).csv(f"{RAW_PATH}/order_items")
orders.show(30, truncate=False)


## Normalize, score and inspect
BR001–BR004 implement conditional completeness, nonnegative totals and header arithmetic. O011 and O012 show missing business evidence; O013 shows an arithmetic disagreement.


In [ ]:
typed = (
    orders.withColumn("customer_id_clean", F.trim("customer_id"))
    .withColumn("status_clean", F.upper(F.trim("status")))
    .withColumn("qty", F.expr("try_cast(quantity as int)"))
    .withColumn("price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("discount_value", F.expr("try_cast(discount as decimal(8,2))"))
    .withColumn("total", F.expr("try_cast(order_total as decimal(18,2))"))
    .withColumn("event_date", F.expr("try_cast(order_date as date)"))
    .withColumn("shipped_on", F.expr("try_cast(shipped_date as date)"))
    .withColumn("arrived_on", F.expr("try_cast(arrival_date as date)"))
)
# Reference tables are deduplicated for membership joins, not as a silent repair.
# A separate customer-key check still exposes duplicate reference records.
customer_keys = (
    customers.select(F.col("customer_id").alias("customer_id_clean"))
    .distinct()
    .withColumn("known_customer", F.lit(True))
)
product_keys = (
    products.select("product_id").distinct().withColumn("known_product", F.lit(True))
)
typed = (
    typed.join(customer_keys, "customer_id_clean", "left")
    .join(product_keys, "product_id", "left")
    .withColumn("key_count", F.count("*").over(Window.partitionBy("order_id")))
)

# A predicate means PASS. NULL is a failure unless the rule explicitly permits it.
rules = [
    (
        "DQ000",
        "parseable",
        "raw_text",
        "Malformed JSON",
        F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull(),
    ),
    (
        "DQ001",
        "customer present",
        "customer_id",
        "Missing or blank customer",
        F.length("customer_id_clean") > 0,
    ),
    (
        "DQ002",
        "positive integer quantity",
        "quantity",
        "Not an integer in 1..1000",
        F.col("qty").between(1, 1000),
    ),
    (
        "DQ003",
        "nonnegative price",
        "unit_price",
        "Invalid or negative price",
        F.col("price") >= 0,
    ),
    (
        "DQ004",
        "allowed status",
        "status",
        "Unknown status",
        F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED"),
    ),
    (
        "DQ005",
        "unique order key",
        "order_id",
        "Duplicate business key; quarantine all copies",
        F.col("key_count") == 1,
    ),
    (
        "DQ006",
        "event window",
        "order_date",
        "Invalid, future, old or outside daily window",
        F.col("event_date") == F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1),
    ),
    (
        "DQ007",
        "known customer",
        "customer_id",
        "Customer reference not found",
        F.coalesce(F.col("known_customer"), F.lit(False)),
    ),
    (
        "DQ008",
        "known product",
        "product_id",
        "Product reference not found",
        F.coalesce(F.col("known_product"), F.lit(False)),
    ),
    (
        "DQ009",
        "discount range",
        "discount",
        "Discount outside 0..100",
        F.col("discount_value").between(0, 100),
    ),
    (
        "BR001",
        "shipment date",
        "shipped_date",
        "SHIPPED requires valid shipped_date",
        (F.col("status_clean") != "SHIPPED") | F.col("shipped_on").isNotNull(),
    ),
    (
        "BR002",
        "cancellation reason",
        "cancellation_reason",
        "CANCELLED requires reason",
        (F.col("status_clean") != "CANCELLED")
        | (F.length(F.trim("cancellation_reason")) > 0),
    ),
    (
        "BR003",
        "nonnegative total",
        "order_total",
        "Invalid or negative order total",
        F.col("total") >= 0,
    ),
    (
        "BR004",
        "order arithmetic",
        "order_total",
        "Header differs from quantity times unit price",
        F.abs(F.col("total") - F.col("qty") * F.col("price"))
        <= F.lit("0.01").cast("decimal(18,2)"),
    ),
]
failure_structs = [
    F.when(
        ~F.coalesce(predicate, F.lit(False)),
        F.struct(
            F.lit(rule_id).alias("dq_rule_id"),
            F.lit(name).alias("dq_rule_name"),
            F.lit(column).alias("dq_column"),
            F.lit(reason).alias("dq_reason"),
        ),
    )
    for rule_id, name, column, reason, predicate in rules
]
scored = (
    typed.withColumn(
        "dq_failures", F.filter(F.array(*failure_structs), lambda x: x.isNotNull())
    )
    .withColumn(
        "dq_status", F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID")
    )
    .withColumn("pipeline_run_id", F.lit(RUN_ID))
    .withColumn("processing_timestamp", F.current_timestamp())
    .cache()
)
scored.count()
valid = scored.filter("dq_status = 'VALID'")
rejected = scored.filter("dq_status = 'INVALID'")
valid.select("order_id", "quantity", "status", "dq_status").show(truncate=False)
rejected.select("order_id", "source_row_id", "dq_failures").show(30, truncate=False)


## Header totals versus item totals
One order can have many items. Sum line amounts by order_id before joining headers. Do not join raw items to headers and then count orders. Invalid item amounts must not silently disappear inside sum: track invalid lines separately.


In [ ]:
item_values = (
    items.withColumn("item_qty", F.expr("try_cast(quantity as int)"))
    .withColumn("item_price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("line_amount", F.col("item_qty") * F.col("item_price"))
)
line_ok = (
    (F.col("item_qty") > 0)
    & (F.col("item_price") >= 0)
    & F.col("line_amount").isNotNull()
)
item_totals = item_values.groupBy("order_id").agg(
    F.sum("line_amount").alias("item_total"),
    F.sum((~F.coalesce(line_ok, F.lit(False))).cast("int")).alias("bad_item_count"),
)
comparison = typed.join(item_totals, "order_id", "left").withColumn(
    "BR005_pass",
    F.coalesce(
        (F.col("bad_item_count") == 0)
        & (
            F.abs(F.col("total") - F.col("item_total"))
            <= F.lit("0.01").cast("decimal(18,2)")
        ),
        F.lit(False),
    ),
)
comparison.filter("BR005_pass").select("order_id", "total", "item_total").show()
comparison.filter("not BR005_pass").select(
    "order_id", "total", "item_total", "bad_item_count"
).show()
items.join(orders.select("order_id").distinct(), "order_id", "left_anti").show()


## Discount policy belongs to the product contract
A discounted selling price may not exceed the original price. Null amounts fail the contract rather than passing through SQL’s unknown truth value.


In [ ]:
p = products.withColumn(
    "original", F.expr("try_cast(original_price as decimal(18,2))")
).withColumn("discounted", F.expr("try_cast(discounted_price as decimal(18,2))"))
p = p.withColumn(
    "BR006_pass",
    F.coalesce(
        (F.col("discounted") >= 0) & (F.col("discounted") <= F.col("original")),
        F.lit(False),
    ),
)
p.filter("BR006_pass").show()
p.filter("not BR006_pass").show()


## Exercise
The shop adds tax and shipping. Update the total formula and its tolerance as a business contract rather than loosening all numeric checks. Decide how refunds and cancelled orders should affect financial control totals.
